# LeetCode 931 — Minimum Falling Path Sum bằng Python

Link đề: https://leetcode.com/problems/minimum-falling-path-sum/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Dynamic Programming (quy hoạch động), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        ...
```

Ghi chú: một **đường rơi (falling path)** bắt đầu ở một ô bất kỳ của hàng đầu và ở mỗi bước đi xuống hàng kế tiếp tới ô **ngay dưới**, **chéo trái dưới** hoặc **chéo phải dưới**. Đây là quy hoạch động kinh điển trên lưới: `dp[r][c] = matrix[r][c] + min(dp[r-1][c-1], dp[r-1][c], dp[r-1][c+1])` (chỉ lấy các ô nằm trong lưới). Cạm bẫy riêng của Python: chỉ số **âm** như `c - 1 = -1` **không gây lỗi** mà quay về cột cuối!

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm tổng nhỏ nhất của một đường đi từ hàng đầu xuống hàng cuối, mỗi bước xuống một hàng và lệch cột không quá `1`.
2. Nắm các công cụ Python cần thiết: lưới 2 chiều, `min` nhiều đối số, cắt biên bằng `max(0, c - 1)` và `min(n, c + 2)`, `functools.cache`, và mảng cuốn chiếu (rolling array).
3. Hiểu nguyên lý quy hoạch động: trạng thái `dp[r][c]` = tổng nhỏ nhất của đường rơi kết thúc ở `(r, c)`; chuyển trạng thái từ hàng trên; đáp án `min(dp[n-1])`.
4. Đi qua 4 cách: duyệt mọi đường (brute `3^n`), đệ quy có ghi nhớ, bảng 2 chiều từ dưới lên, và mảng cuốn chiếu `O(n)` bộ nhớ (cách nộp chính); đối chiếu bằng hai oracle độc lập.
5. Tránh các lỗi kinh điển: tham lam từng bước, quên cắt biên cột (chỉ số `-1` quay vòng), khởi tạo đáp án `0` khi có số âm, và chỉ lấy một trong ba ô trên.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `matrix` cỡ `n x n`, trả về **tổng nhỏ nhất** của một **đường rơi** đi qua `matrix`.

Đường rơi bắt đầu từ **một phần tử bất kỳ ở hàng đầu** và ở mỗi bước chọn phần tử ở hàng kế tiếp ngay phía dưới hoặc chéo trái/phải. Cụ thể, từ vị trí `(row, col)`, phần tử kế tiếp là `(row + 1, col - 1)`, `(row + 1, col)` hoặc `(row + 1, col + 1)`.

**Ví dụ 1**

- Input: `matrix = [[2,1,3],[6,5,4],[7,8,9]]`
- Output: `13`
- Giải thích: hai đường rơi cho tổng nhỏ nhất `13`: `1 -> 5 -> 7` và `1 -> 4 -> 8`.

**Ví dụ 2**

- Input: `matrix = [[-19,57],[-40,-5]]`
- Output: `-59` (đường `-19 -> -40`)

**Ràng buộc:**

- `n == matrix.length == matrix[i].length`
- `1 <= n <= 100`
- `-100 <= matrix[i][j] <= 100`

In [1]:
from typing import List
from functools import cache
import random, time

examples = [
    ([[2, 1, 3], [6, 5, 4], [7, 8, 9]], 13),
    ([[-19, 57], [-40, -5]], -59),
]
for m, expected in examples:
    print(f"matrix={m} kỳ vọng={expected}")

matrix=[[2, 1, 3], [6, 5, 4], [7, 8, 9]] kỳ vọng=13
matrix=[[-19, 57], [-40, -5]] kỳ vọng=-59


## 3. Kiến thức nền

### 3.1. Các ô kế tiếp trong lưới

Từ `(r, c)` xuống hàng `r + 1` có tối đa `3` ô: cột `c - 1`, `c`, `c + 1`, nhưng chỉ những cột nằm trong `[0, n - 1]`. Ở cột `0` chỉ có `2` ô (`c`, `c + 1`), ở cột `n - 1` chỉ có `2` ô (`c - 1`, `c`), với `n = 1` chỉ có `1` ô.

In [2]:
def next_cols(c: int, n: int) -> List[int]:
    return list(range(max(0, c - 1), min(n, c + 2)))

for c in range(4):
    print(f"n=4, cột {c} -> các cột ở hàng dưới: {next_cols(c, 4)}")
print("n=1:", next_cols(0, 1))

n=4, cột 0 -> các cột ở hàng dưới: [0, 1]
n=4, cột 1 -> các cột ở hàng dưới: [0, 1, 2]
n=4, cột 2 -> các cột ở hàng dưới: [1, 2, 3]
n=4, cột 3 -> các cột ở hàng dưới: [2, 3]
n=1: [0]


### 3.2. Chỉ số âm trong Python

`row[-1]` là phần tử **cuối** của `row`, không phải lỗi. Vì vậy `dp[c - 1]` với `c = 0` ngầm lấy cột cuối, cho kết quả sai mà không báo lỗi: cần **cắt biên** tường minh.

In [3]:
row = [10, 20, 30]
print("row[-1] =", row[-1], "(cột cuối, không lỗi)")
c = 0
print("min(row[c - 1], row[c], row[c + 1]) =", min(row[c - 1], row[c], row[c + 1]), "| nhưng cột -1 không tồn tại thật")

row[-1] = 30 (cột cuối, không lỗi)
min(row[c - 1], row[c], row[c + 1]) = 10 | nhưng cột -1 không tồn tại thật


### 3.3. Quy hoạch động từ hàng trên xuống

`dp[r][c]` = tổng nhỏ nhất của một đường rơi từ hàng đầu tới `(r, c)`. Hàng đầu: `dp[0][c] = matrix[0][c]`. Hàng `r > 0`: `dp[r][c] = matrix[r][c] + min(dp[r-1][c'] for c' in next-cột-trên-hợp-lệ)`. Đáp án: `min(dp[n-1])`.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `[[2,1,3],[6,5,4],[7,8,9]]`.

- Hàng 0: `dp = [2, 1, 3]`.
- Hàng 1: `dp[1][0] = 6 + min(2, 1) = 7`; `dp[1][1] = 5 + min(2, 1, 3) = 6`; `dp[1][2] = 4 + min(1, 3) = 5`. `dp = [7, 6, 5]`.
- Hàng 2: `dp[2][0] = 7 + min(7, 6) = 13`; `dp[2][1] = 8 + min(7, 6, 5) = 13`; `dp[2][2] = 9 + min(6, 5) = 14`. `dp = [13, 13, 14]`.

Đáp án `min(13, 13, 14) = 13`.

## 5. Cách 1 — Brute Force: duyệt mọi đường rơi

**Ý tưởng.** Với mỗi cột xuất phát ở hàng đầu, đệ quy thử cả `3` bước kế tiếp (nếu trong lưới) và lấy tổng nhỏ nhất. Số đường là `n * 3^(n-1)`. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [4]:
def falling_brute(matrix: List[List[int]]) -> int:
    n = len(matrix)

    def go(r: int, c: int) -> int:
        if r == n - 1:
            return matrix[r][c]
        return matrix[r][c] + min(go(r + 1, nc) for nc in next_cols(c, n))

    return min(go(0, c) for c in range(n))

for m, expected in examples:
    got = falling_brute(m)
    print(m, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[2, 1, 3], [6, 5, 4], [7, 8, 9]] -> 13 OK
[[-19, 57], [-40, -5]] -> -59 OK


**Độ phức tạp và nút thắt.** `O(n * 3^n)`: với `n = 100` là không thể. Xem thời gian:

In [5]:
rng = random.Random(931)
for n in (4, 6, 8, 10, 12):
    m = [[rng.randint(-100, 100) for _ in range(n)] for _ in range(n)]
    t0 = time.perf_counter()
    falling_brute(m)
    print(f"n={n:2}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n= 4:       0.0 ms
n= 6:       0.4 ms
n= 8:       5.0 ms
n=10:      57.2 ms


n=12:     640.2 ms


Thời gian tăng khoảng `3` lần mỗi hàng thêm vào. Nút thắt: nhiều đường **cùng đi qua** một ô `(r, c)` và ta tính lại phần đường còn lại từ ô đó rất nhiều lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Bài toán con chồng lấn.** "Tổng nhỏ nhất của đường rơi **kết thúc ở** `(r, c)`" chỉ phụ thuộc vào ba ô phía trên. Chỉ có `n^2` trạng thái `(r, c)`, mỗi trạng thái tính trong `O(1)`: tổng `O(n^2)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy **từ dưới lên** (kết thúc ở `(r, c)`) có ghi nhớ |
| 3 | Bảng 2 chiều `dp` tính từ hàng đầu xuống |
| 4 | Mảng cuốn chiếu: chỉ giữ hàng trước (`O(n)` bộ nhớ) (nộp) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có ghi nhớ

**Giải thích từng đoạn.**

- `f(r, c)`: tổng nhỏ nhất của đường rơi kết thúc ở `(r, c)`.
- `f(0, c) = matrix[0][c]`; `f(r, c) = matrix[r][c] + min(f(r - 1, c') for c' in next_cols(c, n))` (các cột trên hợp lệ chính là các cột `c - 1, c, c + 1` nằm trong lưới).
- Đáp án `min(f(n - 1, c))`.
- `@cache` ghi nhớ; chiều sâu đệ quy `n <= 100` nên an toàn.

In [6]:
def falling_memo(matrix: List[List[int]]) -> int:
    n = len(matrix)

    @cache
    def f(r: int, c: int) -> int:
        if r == 0:
            return matrix[0][c]
        return matrix[r][c] + min(f(r - 1, pc) for pc in range(max(0, c - 1), min(n, c + 2)))

    return min(f(n - 1, c) for c in range(n))

for m, expected in examples:
    got = falling_memo(m)
    print(m, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[2, 1, 3], [6, 5, 4], [7, 8, 9]] -> 13 OK
[[-19, 57], [-40, -5]] -> -59 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        from functools import cache
        n = len(matrix)

        @cache
        def f(r: int, c: int) -> int:
            if r == 0:
                return matrix[0][c]
            return matrix[r][c] + min(f(r - 1, pc) for pc in range(max(0, c - 1), min(n, c + 2)))

        return min(f(n - 1, c) for c in range(n))

sol_memo = Solution().minFallingPathSum
assert [sol_memo([r[:] for r in m]) for m, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n^2)` thời gian (`n^2` trạng thái, mỗi trạng thái 3 phép), `O(n^2)` bộ nhớ (bộ nhớ đệm) và `O(n)` ngăn xếp đệ quy.

---

### 7.2. Cách 3 — Bảng 2 chiều từ trên xuống

**Ý tưởng khác.** Thay đệ quy bằng vòng lặp: `dp = [[0] * n for _ in range(n)]`; hàng `0` sao chép từ `matrix`; mỗi hàng sau tính từ hàng trước theo công thức. Đáp án `min(dp[-1])`.

**Giải thích từng đoạn.** `lo = max(0, c - 1)`, `hi = min(n - 1, c + 1)` là cột trên hợp lệ; `min(dp[r - 1][lo:hi + 1])` lấy nhỏ nhất của ba (hoặc hai/một) ô.

In [8]:
def falling_dp2d(matrix: List[List[int]]) -> int:
    n = len(matrix)
    dp = [[0] * n for _ in range(n)]
    dp[0] = matrix[0][:]
    for r in range(1, n):
        for c in range(n):
            lo, hi = max(0, c - 1), min(n - 1, c + 1)
            dp[r][c] = matrix[r][c] + min(dp[r - 1][lo:hi + 1])
    return min(dp[-1])

for m, expected in examples:
    got = falling_dp2d(m)
    print(m, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[2, 1, 3], [6, 5, 4], [7, 8, 9]] -> 13 OK
[[-19, 57], [-40, -5]] -> -59 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        n = len(matrix)
        dp = [[0] * n for _ in range(n)]
        dp[0] = matrix[0][:]
        for r in range(1, n):
            for c in range(n):
                lo, hi = max(0, c - 1), min(n - 1, c + 1)
                dp[r][c] = matrix[r][c] + min(dp[r - 1][lo:hi + 1])
        return min(dp[-1])

sol_dp2d = Solution().minFallingPathSum
assert [sol_dp2d([r[:] for r in m]) for m, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n^2)` thời gian, `O(n^2)` bộ nhớ.

---

### 7.3. Cách 4 — Mảng cuốn chiếu (cách nộp chính)

**Ý tưởng khác.** Hàng `r` chỉ cần hàng `r - 1`, nên giữ **một** mảng `dp` độ dài `n`. Mỗi hàng tạo mảng `new_dp` từ `dp` rồi gán lại.

**Giải thích từng đoạn.**

- `dp = matrix[0][:]`.
- Với mỗi hàng `r >= 1`: `new_dp[c] = matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)])` (lát cắt đã được **cắt biên**, không có chỉ số âm).
- Đáp án `min(dp)` sau hàng cuối.

In [10]:
def falling_rolling(matrix: List[List[int]]) -> int:
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
    return min(dp)

for m, expected in examples:
    got = falling_rolling(m)
    print(m, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[2, 1, 3], [6, 5, 4], [7, 8, 9]] -> 13 OK
[[-19, 57], [-40, -5]] -> -59 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        n = len(matrix)
        dp = matrix[0][:]
        for r in range(1, n):
            dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
        return min(dp)

sol_roll = Solution().minFallingPathSum
assert [sol_roll([r[:] for r in m]) for m, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n^2)` thời gian, `O(n)` bộ nhớ phụ.

## 8. TRACE từng bước

### 8.1. Cách 4 cho ví dụ 1 (từng hàng)

In [12]:
def trace_rolling(matrix):
    n = len(matrix)
    dp = matrix[0][:]
    print("hàng 0:", dp)
    for r in range(1, n):
        new = []
        for c in range(n):
            lo, hi = max(0, c - 1), min(n, c + 2)
            window = dp[lo:hi]
            new.append(matrix[r][c] + min(window))
            print(f"  (r={r}, c={c}): matrix={matrix[r][c]}, các ô trên {window} -> min {min(window)} -> {new[-1]}")
        dp = new
        print(f"hàng {r}:", dp)
    return min(dp)

assert trace_rolling(examples[0][0]) == 13

hàng 0: [2, 1, 3]
  (r=1, c=0): matrix=6, các ô trên [2, 1] -> min 1 -> 7
  (r=1, c=1): matrix=5, các ô trên [2, 1, 3] -> min 1 -> 6
  (r=1, c=2): matrix=4, các ô trên [1, 3] -> min 1 -> 5
hàng 1: [7, 6, 5]
  (r=2, c=0): matrix=7, các ô trên [7, 6] -> min 6 -> 13
  (r=2, c=1): matrix=8, các ô trên [7, 6, 5] -> min 5 -> 13
  (r=2, c=2): matrix=9, các ô trên [6, 5] -> min 5 -> 14
hàng 2: [13, 13, 14]


### 8.2. Ví dụ 2 với số âm

`[[-19, 57], [-40, -5]]`: hàng 0 `dp = [-19, 57]`; hàng 1: `dp[0] = -40 + min(-19, 57) = -59`, `dp[1] = -5 + min(-19, 57) = -24`; đáp án `min(-59, -24) = -59`.

In [13]:
assert trace_rolling(examples[1][0]) == -59

hàng 0: [-19, 57]
  (r=1, c=0): matrix=-40, các ô trên [-19, 57] -> min -19 -> -59
  (r=1, c=1): matrix=-5, các ô trên [-19, 57] -> min -19 -> -24
hàng 1: [-59, -24]


### 8.3. Truy vết đường đi (Cách 3 với `parent`)

Ta lưu cột trước đó cho mỗi `dp[r][c]` để dựng lại đường rơi tối ưu.

In [14]:
def falling_path(matrix):
    n = len(matrix)
    dp = [row[:] for row in matrix]
    parent = [[None] * n for _ in range(n)]
    for r in range(1, n):
        for c in range(n):
            best = min(range(max(0, c - 1), min(n, c + 2)), key=lambda pc: dp[r - 1][pc])
            parent[r][c] = best
            dp[r][c] = matrix[r][c] + dp[r - 1][best]
    c = min(range(n), key=lambda cc: dp[-1][cc])
    total = dp[-1][c]
    path = [c]
    for r in range(n - 1, 0, -1):
        c = parent[r][c]
        path.append(c)
    return total, path[::-1]

total, path = falling_path(examples[0][0])
print("tổng", total, "| cột theo từng hàng:", path, "| giá trị:", [examples[0][0][r][c] for r, c in enumerate(path)])
assert total == 13

tổng 13 | cột theo từng hàng: [1, 1, 0] | giá trị: [1, 5, 7]


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (mảng cuốn chiếu): `O(n^2)` thời gian, chỉ `O(n)` bộ nhớ phụ, lát cắt đã cắt biên nên không lo chỉ số âm.

```python
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        n = len(matrix)
        dp = matrix[0][:]
        for r in range(1, n):
            dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
        return min(dp)
```

In [15]:
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        n = len(matrix)
        dp = matrix[0][:]
        for r in range(1, n):
            dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
        return min(dp)

sol = Solution()
for m, expected in examples:
    got = sol.minFallingPathSum([r[:] for r in m])
    print(m, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[[2, 1, 3], [6, 5, 4], [7, 8, 9]] -> 13 OK
[[-19, 57], [-40, -5]] -> -59 OK


## 10. Vì sao đúng

**(a) Cấu trúc con tối ưu.** Gọi `f(r, c)` là tổng nhỏ nhất của đường rơi từ hàng `0` tới `(r, c)`. Đường rơi tối ưu tới `(r, c)` có ô trước đó là một trong các `(r - 1, c')` với `c' in {c - 1, c, c + 1}` nằm trong lưới; phần đường tới `(r - 1, c')` cũng phải tối ưu (nếu không, thay bằng đường tốt hơn sẽ giảm tổng). Vậy `f(r, c) = matrix[r][c] + min f(r - 1, c')`, `f(0, c) = matrix[0][c]`.

**(b) Đáp án.** Đường rơi kết thúc ở một ô bất kỳ của hàng cuối, nên đáp án là `min f(n - 1, c)`; đường bắt đầu ở hàng đầu được bảo đảm bởi điều kiện cơ sở.

**(c) Cuốn chiếu.** `f(r, ·)` chỉ phụ thuộc `f(r - 1, ·)`, nên chỉ cần giữ một hàng.

**(d) Số âm.** Công thức dùng `min` nên đúng với số âm; đáp án có thể âm (ví dụ 2). Không được khởi tạo đáp án bằng `0`.

**Kiểm chứng bằng máy.** Hai oracle **độc lập** với các cách DP: (1) **duyệt mọi đường rơi** (`n <= 7`); (2) **thư giãn cạnh trên đồ thị có hướng** (DAG): dựng danh sách cạnh tường minh `(r, c) -> (r + 1, c')` và thư giãn khoảng cách theo thứ tự hàng (khác hẳn công thức hồi quy), dùng cho `n` tới `100`.

In [16]:
def oracle_edges(matrix: List[List[int]]) -> int:
    n = len(matrix)
    edges = {}
    for r in range(n - 1):
        for c in range(n):
            edges[(r, c)] = [(r + 1, nc) for nc in (c - 1, c, c + 1) if 0 <= nc < n]
    INF = float("inf")
    dist = {(r, c): INF for r in range(n) for c in range(n)}
    for c in range(n):
        dist[(0, c)] = matrix[0][c]
    for r in range(n - 1):
        for c in range(n):
            for (tr, tc) in edges[(r, c)]:
                cand = dist[(r, c)] + matrix[tr][tc]
                if cand < dist[(tr, tc)]:
                    dist[(tr, tc)] = cand
    return min(dist[(n - 1, c)] for c in range(n))

rng = random.Random(1931)
for _ in range(300):
    n = rng.randint(1, 7)
    m = [[rng.randint(-100, 100) for _ in range(n)] for _ in range(n)]
    assert falling_brute(m) == oracle_edges(m), m
print("duyệt mọi đường và thư giãn cạnh khớp nhau trên 300 ma trận ngẫu nhiên")

duyệt mọi đường và thư giãn cạnh khớp nhau trên 300 ma trận ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi đường rơi | `O(n * 3^n)` | `O(n)` |
| 2. Đệ quy + ghi nhớ | `O(n^2)` | `O(n^2)` |
| 3. Bảng 2 chiều | `O(n^2)` | `O(n^2)` |
| 4. Cuốn chiếu | `O(n^2)` | `O(n)` |

Đo thử với `n = 100` (cỡ tối đa):

In [17]:
rng = random.Random(11)
n = 100
m = [[rng.randint(-100, 100) for _ in range(n)] for _ in range(n)]
ref = oracle_edges(m)
for name, fn in (("Cách 2 ghi nhớ", sol_memo), ("Cách 3 bảng 2 chiều", sol_dp2d), ("Cách 4 cuốn chiếu", sol_roll), ("nộp", Solution().minFallingPathSum)):
    t0 = time.perf_counter()
    got = fn([r[:] for r in m])
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:20} n={n}: {dt:8.2f} ms (đáp án {got})")

Cách 2 ghi nhớ       n=100:    16.49 ms (đáp án -6528)
Cách 3 bảng 2 chiều  n=100:     4.64 ms (đáp án -6528)


Cách 4 cuốn chiếu    n=100:     4.69 ms (đáp án -6528)
nộp                  n=100:     4.11 ms (đáp án -6528)


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với hai oracle, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [18]:
def solutions_list():
    return {
        "memo": falling_memo,
        "dp2d": falling_dp2d,
        "rolling": falling_rolling,
        "sol_memo": sol_memo,
        "sol_dp2d": sol_dp2d,
        "sol_roll": sol_roll,
        "final": Solution().minFallingPathSum,
    }

def run_all(m, expected=None, brute_limit=7):
    ref = falling_brute(m) if len(m) <= brute_limit else oracle_edges(m)
    assert oracle_edges(m) == ref
    for name, fn in solutions_list().items():
        assert fn([r[:] for r in m]) == ref, (name, m)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", m, expected, ref)
    return ref

# 12.1 ví dụ đề
for m, expected in examples:
    assert run_all(m, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[5]], 5),                                         # n = 1
    ([[-7]], -7),                                       # n = 1 âm
    ([[1, 2], [3, 4]], 4),                              # 1 -> 3
    ([[-1, -2], [-3, -4]], -6),                         # -2 -> -4
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 0),             # toàn 0
    ([[100, 100], [100, 100]], 200),                    # toàn lớn
    ([[-100] * 3] * 3, -300),                           # toàn nhỏ nhất
    ([[1, 100, 100], [100, 100, 1], [100, 1, 100]], 102),   # dp: [1,100,100] -> [101,101,101] -> [201,102,201]
    ([[2, 1, 3], [6, 5, 4], [7, 8, 9]], 13),
    ([[5, 3, 8], [4, 9, 1], [7, 2, 6]], 6),             # 3 -> 1 -> 2
]
for m, expected in edge:
    run_all(m, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_greedy(matrix):                          # tham lam: mỗi bước đi tới ô nhỏ nhất trong ba ô dưới
    n = len(matrix)
    c = min(range(n), key=lambda cc: matrix[0][cc])
    total = matrix[0][c]
    for r in range(1, n):
        c = min(range(max(0, c - 1), min(n, c + 2)), key=lambda cc: matrix[r][cc])
        total += matrix[r][c]
    return total

def wrong_negative_index(matrix):                  # quên cắt biên cột: chỉ số -1 quay vòng, c + 1 có thể tràn
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        new = []
        for c in range(n):
            options = [dp[c - 1], dp[c]]               # dp[-1] là cột cuối!
            if c + 1 < n:
                options.append(dp[c + 1])
            new.append(matrix[r][c] + min(options))
        dp = new
    return min(dp)

def wrong_two_neighbors(matrix):                   # chỉ xét trái-trên và trên (bỏ chéo phải)
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        dp = [matrix[r][c] + min(dp[max(0, c - 1): c + 1]) for c in range(n)]
    return min(dp)

def wrong_init_zero(matrix):                       # đáp án khởi tạo 0 trước khi lấy min (sai khi mọi đường dương)
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
    return min(0, min(dp))

def wrong_first_column_end(matrix):                # lấy dp ở cột 0 của hàng cuối thay vì min cả hàng
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
    return dp[0]

def find_counter(wrong, tries=20000, seed=8):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 5)
        m = [[rng.randint(-9, 9) for _ in range(n)] for _ in range(n)]
        if wrong(m) != falling_brute(m):
            return m
    return None

counters = {}
for name, fn in (("tham lam từng bước", wrong_greedy), ("chỉ số -1 quay vòng", wrong_negative_index),
                 ("bỏ ô chéo phải", wrong_two_neighbors), ("khởi tạo đáp án 0", wrong_init_zero),
                 ("lấy dp cột 0 của hàng cuối", wrong_first_column_end)):
    m = find_counter(fn)
    assert m is not None, name
    counters[name] = m
    print(f"{name:28} m={m}: sai -> {fn(m)}, đúng -> {falling_brute(m)}")
    run_all(m)
print("12.3 phản ví dụ: OK")

tham lam từng bước           m=[[5, 3, 6, 9], [-3, 3, -7, 6], [-2, -9, -1, 7], [4, 6, 3, -6]]: sai -> -10, đúng -> -11
chỉ số -1 quay vòng          m=[[5, 4, 4, -7, 9], [-7, 0, -6, -8, -7], [-6, 7, 7, 2, -5], [7, 6, -5, -7, -3], [-9, -5, 1, 3, 9]]: sai -> -32, đúng -> -26
bỏ ô chéo phải               m=[[7, -3, 9, -5], [-7, 8, -8, 6], [-3, -5, 9, 5], [9, 5, 0, 8]]: sai -> -5, đúng -> -18
khởi tạo đáp án 0            m=[[5]]: sai -> 0, đúng -> 5
lấy dp cột 0 của hàng cuối   m=[[5, 3, 6, 9], [-3, 3, -7, 6], [-2, -9, -1, 7], [4, 6, 3, -6]]: sai -> -9, đúng -> -11
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ: n <= 7, mọi cách so với duyệt mọi đường và thư giãn cạnh
rng = random.Random(4931)
for _ in range(1000):
    n = rng.randint(1, 7)
    lo, hi = rng.choice([(-100, 100), (-3, 3), (0, 100), (-100, 0)])
    m = [[rng.randint(lo, hi) for _ in range(n)] for _ in range(n)]
    run_all(m)
print("12.4 ngẫu nhiên n <= 7 (1000 ca): OK")

# 12.5 trung bình: n tới 40 so với thư giãn cạnh
rng = random.Random(5931)
for _ in range(100):
    n = rng.randint(8, 40)
    lo, hi = rng.choice([(-100, 100), (-1, 1), (0, 100)])
    m = [[rng.randint(lo, hi) for _ in range(n)] for _ in range(n)]
    run_all(m)
print("12.5 ngẫu nhiên n tới 40 (100 ca): OK")

# 12.6 lớn n = 100 (đúng ràng buộc) nhiều dạng
rng = random.Random(6931)
n = 100
shapes = {
    "ngẫu nhiên": [[rng.randint(-100, 100) for _ in range(n)] for _ in range(n)],
    "toàn -100": [[-100] * n for _ in range(n)],
    "toàn 100": [[100] * n for _ in range(n)],
    "toàn 0": [[0] * n for _ in range(n)],
    "đường chéo rẻ": [[-100 if r == c else 100 for c in range(n)] for r in range(n)],
    "cột giữa rẻ": [[-100 if c == n // 2 else 100 for c in range(n)] for r in range(n)],
}
expected_big = {"toàn -100": -100 * n, "toàn 100": 100 * n, "toàn 0": 0, "đường chéo rẻ": -100 * n, "cột giữa rẻ": -100 * n}
for name, m in shapes.items():
    ref = oracle_edges(m)
    for fname, fn in solutions_list().items():
        assert fn([r[:] for r in m]) == ref, (name, fname)
    if name in expected_big:
        assert ref == expected_big[name], name
    print(f"  {name:14} đáp án {ref}")
print("12.6 n = 100: OK")

# 12.7 tính chất: lật gương trái-phải không đổi đáp án; cộng hằng số c vào mọi ô tăng đáp án n * c
rng = random.Random(7931)
for _ in range(200):
    n = rng.randint(1, 10)
    m = [[rng.randint(-50, 50) for _ in range(n)] for _ in range(n)]
    base = Solution().minFallingPathSum([r[:] for r in m])
    assert Solution().minFallingPathSum([r[::-1] for r in m]) == base
    k = rng.randint(-20, 20)
    assert Solution().minFallingPathSum([[x + k for x in r] for r in m]) == base + n * k
print("12.7 tính chất (gương, cộng hằng số): OK")

12.4 ngẫu nhiên n <= 7 (1000 ca): OK


12.5 ngẫu nhiên n tới 40 (100 ca): OK
  ngẫu nhiên     đáp án -6345
  toàn -100      đáp án -10000


  toàn 100       đáp án 10000
  toàn 0         đáp án 0
  đường chéo rẻ  đáp án -10000
  cột giữa rẻ    đáp án -10000
12.6 n = 100: OK
12.7 tính chất (gương, cộng hằng số): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam từng bước

Chọn ô nhỏ nhất trong ba ô dưới ở mỗi bước có thể dẫn tới một hàng sau rất đắt.

In [21]:
m = counters["tham lam từng bước"]
print("m =", m, "| sai ->", wrong_greedy(m), "| đúng ->", Solution().minFallingPathSum([r[:] for r in m]))
assert wrong_greedy(m) != Solution().minFallingPathSum([r[:] for r in m])

m = [[5, 3, 6, 9], [-3, 3, -7, 6], [-2, -9, -1, 7], [4, 6, 3, -6]] | sai -> -10 | đúng -> -11


Vì sao sai: một lựa chọn nhỏ ở hàng này có thể khóa vào vùng đắt ở hàng sau. **Đúng:** quy hoạch động tính tối ưu cho mọi ô.

### Lỗi 2 — Quên cắt biên: chỉ số `-1` quay vòng

`dp[c - 1]` với `c = 0` lấy `dp[-1]` (cột cuối) **không báo lỗi** và cho kết quả sai.

In [22]:
m = counters["chỉ số -1 quay vòng"]
print("m =", m, "| sai ->", wrong_negative_index(m), "| đúng ->", Solution().minFallingPathSum([r[:] for r in m]))
assert wrong_negative_index(m) != Solution().minFallingPathSum([r[:] for r in m])

m = [[5, 4, 4, -7, 9], [-7, 0, -6, -8, -7], [-6, 7, 7, 2, -5], [7, 6, -5, -7, -3], [-9, -5, 1, 3, 9]] | sai -> -32 | đúng -> -26


Vì sao sai: cột `0` không có ô chéo trái; Python lại cho phép `dp[-1]` nên ô cột cuối bị coi là "chéo trái". **Đúng:** cắt biên tường minh `dp[max(0, c - 1): min(n, c + 2)]`.

### Lỗi 3 — Bỏ một trong ba ô trên

Bỏ ô chéo phải (hoặc trái) làm mất những đường hợp lệ.

In [23]:
m = counters["bỏ ô chéo phải"]
print("m =", m, "| sai ->", wrong_two_neighbors(m), "| đúng ->", Solution().minFallingPathSum([r[:] for r in m]))
assert wrong_two_neighbors(m) != Solution().minFallingPathSum([r[:] for r in m])

m = [[7, -3, 9, -5], [-7, 8, -8, 6], [-3, -5, 9, 5], [9, 5, 0, 8]] | sai -> -5 | đúng -> -18


Vì sao sai: đường tối ưu có thể đi chéo sang phải. **Đúng:** xét cả ba ô trên (đã cắt biên).

### Lỗi 4 — Khởi tạo đáp án bằng `0`

Khi mọi đường đều có tổng dương, `min(0, ...)` trả `0`, sai.

In [24]:
m = counters["khởi tạo đáp án 0"]
print("m =", m, "| sai ->", wrong_init_zero(m), "| đúng ->", Solution().minFallingPathSum([r[:] for r in m]))
assert wrong_init_zero(m) != Solution().minFallingPathSum([r[:] for r in m])

m = [[5]] | sai -> 0 | đúng -> 5


Vì sao sai: đáp án là `min` của các tổng thật; không có "đường rỗng" tổng `0`. **Đúng:** `min(dp)` trực tiếp.

### Lỗi 5 — Chỉ lấy `dp` ở một cột của hàng cuối

Đường rơi có thể kết thúc ở **bất kỳ** cột nào của hàng cuối.

In [25]:
m = counters["lấy dp cột 0 của hàng cuối"]
print("m =", m, "| sai ->", wrong_first_column_end(m), "| đúng ->", Solution().minFallingPathSum([r[:] for r in m]))
assert wrong_first_column_end(m) != Solution().minFallingPathSum([r[:] for r in m])

m = [[5, 3, 6, 9], [-3, 3, -7, 6], [-2, -9, -1, 7], [4, 6, 3, -6]] | sai -> -9 | đúng -> -11


Vì sao sai: bỏ qua các đường kết thúc ở cột khác. **Đúng:** `min(dp[-1])`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đường đi từ hàng đầu xuống hàng cuối, mỗi bước xuống một hàng với vài lựa chọn cột" ⇒ **DP theo hàng**: `dp[r][c] = a[r][c] + min(ô trên hợp lệ)`.
- **Cắt biên tường minh** (`max(0, c - 1)`, `min(n, c + 2)`), nhớ chỉ số `-1` của Python quay vòng.
- **Cuốn chiếu**: chỉ giữ hàng trước.
- Đáp án là `min` của hàng cuối (có thể âm).
- Cùng họ: Triangle (120), Minimum Falling Path Sum II (1289), Cherry Pickup (741), Dungeon Game (174), Unique Paths (62).
- Test nên có: `n = 1`, mọi số âm, mọi số dương, ma trận đồng nhất, `n = 100`.

## 15. Biến thể

1. **Trả về chính đường đi** (bài tập 5): truy vết bằng `parent`.
2. **Đường rơi có tổng lớn nhất:** thay `min` bằng `max`.
3. **Falling Path Sum II (1289):** không được chọn cùng cột ở hai hàng liên tiếp: giữ hai giá trị nhỏ nhất của hàng trước.
4. Bài liên quan: Triangle (120).

Dưới đây là biến thể 2 (tổng lớn nhất) kiểm bằng vét cạn.

In [26]:
def falling_max(matrix):
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        dp = [matrix[r][c] + max(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
    return max(dp)

def falling_max_brute(matrix):
    n = len(matrix)
    def go(r, c):
        if r == n - 1:
            return matrix[r][c]
        return matrix[r][c] + max(go(r + 1, nc) for nc in next_cols(c, n))
    return max(go(0, c) for c in range(n))

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 6)
    m = [[rng.randint(-20, 20) for _ in range(n)] for _ in range(n)]
    assert falling_max(m) == falling_max_brute(m), m
print("biến thể 2 khớp; ví dụ đề (max):", falling_max(examples[0][0]), falling_max(examples[1][0]))

biến thể 2 khớp; ví dụ đề (max): 17 52


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đường | Định nghĩa | `O(n 3^n)` | `O(n)` | Oracle, `n <= 7` |
| 2. Ghi nhớ | Top-down | `O(n^2)` | `O(n^2)` | Dễ viết |
| 3. Bảng 2 chiều | Bottom-up | `O(n^2)` | `O(n^2)` | Cần truy vết |
| 4. Cuốn chiếu | Một hàng | `O(n^2)` | `O(n)` | **Cách nộp chính** |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(next_cols(0, 3), next_cols(1, 3), next_cols(2, 3))
print([10, 20, 30][-1], [10, 20, 30][max(0, -1):2])
print(min([4, 7, 2][0:2]))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `matrix = [[5, 3, 8], [4, 9, 1], [7, 2, 6]]`: ghi tổng nhỏ nhất và một đường rơi (cột theo từng hàng).

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None
hand_path = None              # ví dụ [c0, c1, c2]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành mảng cuốn chiếu.

In [29]:
def exercise_3_rolling(matrix: List[List[int]]) -> int:
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        # TODO: dp mới: với mỗi cột c, matrix[r][c] + min của các ô trên hợp lệ (đã cắt biên)
        pass
    return min(dp)


# for m, expected in examples:
#     assert exercise_3_rolling(m) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_memo(matrix)` bằng đệ quy có ghi nhớ (kết thúc ở `(r, c)`).

In [30]:
def exercise_4_memo(matrix: List[List[int]]) -> int:
    # TODO
    pass


# for m, expected in examples:
#     assert exercise_4_memo(m) == expected

### Bài 5 — Mini challenge: trả về đường rơi tối ưu (khó)

Viết `exercise_5_path(matrix)` trả về một danh sách `path` gồm `n` chỉ số cột (một cho mỗi hàng) mô tả một đường rơi **hợp lệ** có tổng nhỏ nhất. Có nhiều đáp án; ta kiểm tra bằng hàm kiểm tra (hợp lệ + tổng bằng tối ưu).

```python
exercise_5_path([[2, 1, 3], [6, 5, 4], [7, 8, 9]])  -> [1, 1, 0] hoặc [1, 2, 1]
```

In [31]:
def exercise_5_path(matrix: List[List[int]]) -> List[int]:
    # TODO
    pass


# m = [[2, 1, 3], [6, 5, 4], [7, 8, 9]]
# p = exercise_5_path(m)
# assert len(p) == 3 and all(abs(p[i] - p[i + 1]) <= 1 for i in range(2)) and sum(m[r][c] for r, c in enumerate(p)) == 13

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `next_cols(c, n)` cắt biên về `[0, n - 1]`.
- Mức 2: cột `0` có hai ô (`0, 1`); cột giữa có ba; cột cuối có hai.
- Mức 3: `[0, 1] [0, 1, 2] [1, 2]`, `30 [10, 20]`, `4`.

**Bài 2**
- Mức 1: tính dp từng hàng: hàng 0 `[5, 3, 8]`.
- Mức 2: hàng 1: `[4 + 3, 9 + 3, 1 + 3] = [7, 12, 4]`; hàng 2: `[7 + 7, 2 + 4, 6 + 4] = [14, 6, 10]`.
- Mức 3: đáp án `6`, đường cột `[1, 2, 1]` (giá trị `3, 1, 2`).

**Bài 3**
- Mức 1: lát cắt `dp[max(0, c - 1): min(n, c + 2)]`.
- Mức 2: `dp = [matrix[r][c] + min(...) for c in range(n)]`.
- Mức 3: gán lại `dp` cho hàng kế.

**Bài 4**
- Mức 1: `@cache def f(r, c)`.
- Mức 2: `f(0, c) = matrix[0][c]`; `f(r, c) = matrix[r][c] + min(f(r - 1, pc) ...)`.
- Mức 3: đáp án `min(f(n - 1, c) for c in range(n))`.

**Bài 5**
- Mức 1: lưu `parent[r][c]` là cột trước tốt nhất.
- Mức 2: chọn cột cuối có `dp` nhỏ nhất.
- Mức 3: đi ngược `parent` rồi đảo danh sách.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(next_cols(0, 3), next_cols(1, 3), next_cols(2, 3))
print([10, 20, 30][-1], [10, 20, 30][max(0, -1):2])
print(min([4, 7, 2][0:2]))

assert (next_cols(0, 3), next_cols(1, 3), next_cols(2, 3)) == ([0, 1], [0, 1, 2], [1, 2])
assert ([10, 20, 30][-1], [10, 20, 30][max(0, -1):2]) == (30, [10, 20])
assert min([4, 7, 2][0:2]) == 4

[0, 1] [0, 1, 2] [1, 2]
30 [10, 20]
4


### Đáp án Bài 2

Hàng 0: `[5, 3, 8]`; hàng 1: `[7, 12, 4]`; hàng 2: `[14, 6, 10]`; đáp án `6`, đường `3 -> 1 -> 2` (cột `[1, 2, 1]`).

In [33]:
mat = [[5, 3, 8], [4, 9, 1], [7, 2, 6]]
hand_answer_solution = 6
hand_path_solution = [1, 2, 1]
assert Solution().minFallingPathSum([r[:] for r in mat]) == hand_answer_solution == falling_brute(mat)
assert all(abs(hand_path_solution[i] - hand_path_solution[i + 1]) <= 1 for i in range(2))
assert sum(mat[r][c] for r, c in enumerate(hand_path_solution)) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_rolling_solution(matrix: List[List[int]]) -> int:
    n = len(matrix)
    dp = matrix[0][:]
    for r in range(1, n):
        dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
    return min(dp)

for m, expected in examples:
    assert exercise_3_rolling_solution(m) == expected
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 6)
    m = [[rng.randint(-20, 20) for _ in range(n)] for _ in range(n)]
    assert exercise_3_rolling_solution(m) == falling_brute(m)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_memo_solution(matrix: List[List[int]]) -> int:
    n = len(matrix)

    @cache
    def f(r: int, c: int) -> int:
        if r == 0:
            return matrix[0][c]
        return matrix[r][c] + min(f(r - 1, pc) for pc in range(max(0, c - 1), min(n, c + 2)))

    return min(f(n - 1, c) for c in range(n))

for m, expected in examples:
    assert exercise_4_memo_solution(m) == expected
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 6)
    m = [[rng.randint(-20, 20) for _ in range(n)] for _ in range(n)]
    assert exercise_4_memo_solution(m) == falling_brute(m)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_path_solution(matrix: List[List[int]]) -> List[int]:
    return falling_path(matrix)[1]

def check_path(matrix, path):
    n = len(matrix)
    return (len(path) == n and all(0 <= c < n for c in path)
            and all(abs(path[i] - path[i + 1]) <= 1 for i in range(n - 1))
            and sum(matrix[r][c] for r, c in enumerate(path)) == falling_brute(matrix))

m = [[2, 1, 3], [6, 5, 4], [7, 8, 9]]
assert check_path(m, exercise_5_path_solution(m))
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 6)
    mm = [[rng.randint(-20, 20) for _ in range(n)] for _ in range(n)]
    assert check_path(mm, exercise_5_path_solution(mm)), mm
print("Bài 5 OK", exercise_5_path_solution(m))

Bài 5 OK [1, 1, 0]


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (mảng cuốn chiếu)

```python
class Solution:
    def minFallingPathSum(self, matrix: List[List[int]]) -> int:
        n = len(matrix)
        dp = matrix[0][:]                                           # hàng đầu: bắt đầu ở bất kỳ ô nào
        for r in range(1, n):
            # ô trên hợp lệ: cột c - 1, c, c + 1 nằm trong [0, n - 1]  (cắt biên: Python không báo lỗi chỉ số -1)
            dp = [matrix[r][c] + min(dp[max(0, c - 1): min(n, c + 2)]) for c in range(n)]
        return min(dp)                                              # kết thúc ở bất kỳ cột nào của hàng cuối
```

Công thức nhớ nhanh: `dp[r][c] = matrix[r][c] + min(dp[r-1][c-1..c+1])`, đáp án `min(dp[n-1])`.

### Checklist trước khi submit

- [ ] Cắt biên cột tường minh (không dùng `dp[c - 1]` khi `c = 0`)?
- [ ] Xét cả ba ô trên (trái, giữa, phải)?
- [ ] Đáp án là `min` của **cả** hàng cuối (có thể âm, không khởi tạo `0`)?
- [ ] Không dùng tham lam từng bước?
- [ ] Đã thử: `n = 1`, mọi số âm, mọi số dương, `n = 100`?
- [ ] Tên method đúng `minFallingPathSum(self, matrix)` và trả `int`?